# McCartneyTotalHDLRatio

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
4. [Load features](#Load-features)
5. [Load weights into base model](#Load-weights-into-base-model)
6. [Load reference values](#Load-reference-values)
7. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
8. [Check all clock parameters](#Check-all-clock-parameters)
9. [Normal feature ranges](#Normal-feature-ranges)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import json
import math
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.McCartneyTotalHDLRatio)

class McCartneyTotalHDLRatio(LinearReferenceClock):
    pass



In [3]:
model = pya.models.McCartneyTotalHDLRatio()

## Define clock metadata

In [4]:
model.metadata["clock_name"] = "mccartneytotalhdlratio"
model.metadata["data_type"] = "DNA methylation"  # Paper: The predictors use DNA methylation states at CpG sites.
model.metadata["species"] = "Homo sapiens"  # Paper: Generation Scotland is a population-based cohort of human participants.
model.metadata["year"] = 2018
model.metadata["approved_by_author"] = "⌛"
model.metadata["citation"] = "McCartney, D. L., et al. “Epigenetic prediction of complex traits and death.” Genome Biology 19, 136 (2018)."
model.metadata["doi"] = "https://doi.org/10.1186/s13059-018-1514-1"
model.metadata["notes"] = "Whole-blood DNAm LASSO score for total-to-HDL cholesterol ratio, trained in Generation Scotland on an age-, sex-, and ancestry-adjusted phenotype residual and evaluated out of sample in LBC1936."
model.metadata["research_only"] = None
model.metadata["tissue"] = ["whole blood"]  # Paper: Stored DNA from baseline blood samples was used to build the predictors.
model.metadata["predicts"] = ["total-to-HDL cholesterol ratio"]  # Paper: The study developed a DNAm predictor for total-to-HDL cholesterol ratio.
model.metadata["training_target"] = ["total-to-HDL cholesterol ratio"]  # Paper: The total-to-HDL cholesterol ratio phenotype was regressed on age, sex and ten genetic principal components; its residual was the LASSO outcome.
model.metadata["unit"] = ["ratio"]  # Paper: No output transformation is applied; the score retains the dimensionless ratio-residual scale.
model.metadata["model_type"] = "LASSO regression"  # Paper: The glmnet mixing parameter alpha was set to 1, applying a LASSO penalty with tenfold cross-validation.
model.metadata["platform"] = ["Illumina EPIC"]  # Paper: Predictor training used quality-controlled HumanMethylationEPIC blood data; probes absent from 450K were filtered only to enable LBC1936 prediction.
model.metadata["population"] = "adults"  # Paper: The predictors were built on a subset of 5,087 Generation Scotland participants; the parent cohort spans ages 18–99.
model.metadata["journal"] = "Genome Biology"
model.metadata["last_author"] = "Riccardo E. Marioni"
model.metadata["n_features"] = 412
model.metadata["citations"] = 301
model.metadata["citations_date"] = "2026-07-05"


## Download clock dependencies

In [5]:
supplementary_url = "https://static-content.springer.com/esm/art%3A10.1186%2Fs13059-018-1514-1/MediaObjects/13059_2018_1514_MOESM1_ESM.xlsx"
supplementary_file_name = "mccartney_predictors.xlsx"
os.system(f"curl -sL -o {supplementary_file_name} {supplementary_url}")

0

## Load features

In [6]:
# Additional file 1, Table S8 - Total-HDL ratio (McCartney et al. 2018)
coef_df = pd.read_excel('mccartney_predictors.xlsx', sheet_name='Table S8 - Total-HDL ratio')
model.features = coef_df['CpG'].tolist()

## Load weights into base model

In [7]:
# The penalised (LASSO) predictor has no intercept term
weights = torch.tensor(coef_df['Beta'].tolist()).unsqueeze(0).float()
intercept = torch.tensor([0.0]).float()

In [8]:
base_model = pya.models.LinearModel(input_dim=len(model.features))

base_model.linear.weight.data = weights.float()
base_model.linear.bias.data = intercept.float()

model.base_model = base_model

## Load reference values

In [9]:
model.reference_values = None

## Load preprocess and postprocess objects

In [10]:
model.preprocess_name = None
model.preprocess_dependencies = None

In [11]:
model.postprocess_name = None
model.postprocess_dependencies = None

## Check all clock parameters

In [12]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'McCartney, Daniel L., et al. "Epigenetic prediction of complex '
             'traits and death." Genome biology 19.1 (2018): 136.',
 'clock_name': 'mccartneytotalhdlratio',
 'data_type': 'methylation',
 'doi': 'https://doi.org/10.1186/s13059-018-1514-1',
 'notes': None,
 'research_only': None,
 'species': 'Homo sapiens',
 'version': None,
 'year': 2018}
reference_values: None
preprocess_name: None
preprocess_dependencies: None
postprocess_name: None
postprocess_dependencies: None
features: ['cg24576270', 'cg25158622', 'cg06500161', 'cg26033520', 'cg18615457', 'cg10993470', 'cg19250790', 'cg09177238', 'cg22813794', 'cg01511534', 'cg01559436', 'cg22488164', 'cg05391946', 'cg09503194', 'cg14697880', 'cg06136443', 'cg14545305', 'cg26955383', 'cg26800462', 'cg08854185', 'cg14891022', 'cg23719318', 'cg12332083', 'cg0

## Normal feature ranges

In [ ]:
# Units and plausibility ranges come from the package registry, keyed by feature name.
feature_ranges = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
model.feature_units = [record["unit"] for record in feature_ranges]
pd.DataFrame.from_records(feature_ranges).head()

## Basic test

In [ ]:
# Exercise the clock with values in the middle of each feature's expected range.
records = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
midpoints = [
    (record["low"] + record["high"]) / 2 if math.isfinite(record["high"]) else max(record["low"], 1.0)
    for record in records
]
input = torch.tensor([midpoints] * 10, dtype=torch.float64)
model.eval()
model.to(torch.float64)
pred = model(input)
pred

## Save torch model

In [14]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory
<a id="10"></a>

In [15]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)

Deleted file: mccartney_predictors.xlsx
